In [42]:
import os
import numpy as np
from PIL import Image
from tqdm import tqdm

In [49]:
RAW_DIR = "filtered_images_2"
OUTPUT_DIR = "filtered_images"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [50]:
def is_white_pixel(pixel, threshold=240):
    """
    pixel: array [R,G,B]
    threshold: valeur minimale pour considérer comme blanc
    """
    return all(channel >= threshold for channel in pixel)

In [51]:
def white_ratio(image_path, threshold=240):
    image = Image.open(image_path).convert("RGB")
    img_array = np.array(image)

    # Masque pixels blancs
    white_mask = np.all(img_array >= threshold, axis=2)

    white_pixels = np.sum(white_mask)
    total_pixels = img_array.shape[0] * img_array.shape[1]

    return white_pixels / total_pixels

In [52]:
import os
import shutil
import uuid
from tqdm import tqdm

WHITE_THRESHOLD = 0.40
PIXEL_THRESHOLD = 240

processed = 2218

for root, _, files in os.walk(RAW_DIR):
    for filename in tqdm(files, leave=False):
        
        if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        
        path = os.path.join(root, filename)
        
        try:
            ratio = white_ratio(path, threshold=PIXEL_THRESHOLD)
            
            if ratio >= WHITE_THRESHOLD:
                processed += 1
                # Génère un nom unique pour éviter conflits
                ext = os.path.splitext(filename)[1].lower()
                new_name = f"chair_{processed:05d}{ext}"
                
                shutil.copy(path, os.path.join(OUTPUT_DIR, new_name))
        
        except Exception:
            continue

In [53]:
# Nombre d'images dans le dossier de sortie

count = 0
for _, _, files in os.walk(OUTPUT_DIR):
    count += len(files)
print(f"Nombre d'images dans le dossier de sortie : {count}")

Nombre d'images dans le dossier de sortie : 1872


In [54]:
# Gestion des doublons

seen_hashes = set()
for root, _, files in os.walk(OUTPUT_DIR):
    for filename in tqdm(files, leave=False):
        path = os.path.join(root, filename)
        try:
            with Image.open(path) as img:
                img_hash = hash(img.tobytes())
                if img_hash in seen_hashes:
                    os.remove(path)
                else:
                    seen_hashes.add(img_hash)
        except Exception:
            continue

print(f"Nombre d'images après suppression des doublons : {len(seen_hashes)}")

Nombre d'images après suppression des doublons : 1872


# Data augmentation

In [55]:
# Ajout des miroirs des images pour augmenter le dataset

for root, _, files in os.walk(OUTPUT_DIR):
    for filename in tqdm(files, leave=False):
        path = os.path.join(root, filename)
        try:
            with Image.open(path) as img:
                mirrored = img.transpose(Image.FLIP_LEFT_RIGHT)
                mirrored_name = f"mirror_{filename}"
                mirrored.save(os.path.join(OUTPUT_DIR, mirrored_name))
        except Exception:
            continue

In [56]:
# Changer le contraste et la luminiosité
# Ici on fait varier aléatoirement le contraste et la luminiosité entre 0.7 et 1.3 

from PIL import ImageEnhance

for root, _, files in os.walk(OUTPUT_DIR):
    for filename in tqdm(files, leave=False):
        path = os.path.join(root, filename)
        try:
            with Image.open(path) as img:
                # Générer des facteurs de contraste et de luminosité aléatoires
                contrast_factor = np.random.uniform(0.7, 1.3)
                brightness_factor = np.random.uniform(0.7, 1.3)

                # Appliquer le contraste
                enhancer_contrast = ImageEnhance.Contrast(img)
                img_contrasted = enhancer_contrast.enhance(contrast_factor)

                # Appliquer la luminosité
                enhancer_brightness = ImageEnhance.Brightness(img_contrasted)
                img_enhanced = enhancer_brightness.enhance(brightness_factor)

                enhanced_name = f"enhanced_{filename}"
                img_enhanced.save(os.path.join(OUTPUT_DIR, enhanced_name))
        except Exception:
            continue